# Phase 1 — Lesson 05: Chain Rule & Automatic Differentiation
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 05: Chain Rule & Automatic Differentiation
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lesson 04 (Derivatives & Gradients)



# What You'll Learn

---

## Learning Objectives

- Build a minimal autograd engine (Value class) that records operations and computes gradients via reverse-mode autodiff
- Implement forward and backward passes through a computation graph using topological sort
- Construct and train a multi-layer perceptron on XOR using only the from-scratch autograd engine
- Verify autodiff correctness using gradient checking against numerical finite differences


# The Problem

---

You can compute derivatives of simple functions. But a neural network is not a simple function. It is hundreds of functions composed together: matrix multiply, add bias, apply activation, matrix multiply again, softmax, cross-entropy loss. The output is a function of a function of a function.

To train the network, you need the gradient of the loss with respect to every single weight. Doing this by hand is impossible for millions of parameters. Doing it numerically (finite differences) is too slow.

**The chain rule gives you the math. Automatic differentiation gives you the algorithm.** Together they let you compute exact gradients through arbitrary compositions of functions in time proportional to a single forward pass.

This is how PyTorch, TensorFlow, and JAX work. You will build a miniature version from scratch.


# The Chain Rule

---

If `y = f(g(x))`, the derivative of `y` with respect to `x` is:

```
dy/dx = dy/dg · dg/dx = f'(g(x)) · g'(x)
```

Multiply the derivatives along the chain. Each link contributes its local derivative.

Example: `y = sin(x²)`:
```
g(x) = x²        → g'(x) = 2x
f(g) = sin(g)    → f'(g) = cos(g)

dy/dx = cos(x²) · 2x
```

For deeper compositions, the chain extends:
```
y = f(g(h(x)))

dy/dx = f'(g(h(x))) · g'(h(x)) · h'(x)
```

Every layer in a neural network is one link in this chain.


# Computational Graphs

---

A computational graph makes the chain rule visual. Every operation becomes a node. Data flows forward through the graph. Gradients flow backward.

**Forward pass (compute values):**
```
x1 = 2  --->  * (multiply)  --->  + (add)  --->  relu  --->  output y
x2 = 3  --->                   |                |
b = 1   --------------------->                  |
                              a=6      c=7      y=7
```

**Backward pass (compute gradients):**
```
dy/dy = 1  --->  dy/dc = 1  --->  dy/da = 1  --->  dy/dx1 = 3
                              |              |       |
                              |              |       --->  dy/dx2 = 2
                              |              --->  dy/db = 1
                              --->  dy/dc = 1
```

The backward pass applies the chain rule at every node, propagating gradients from output to inputs.


# Forward Mode vs Reverse Mode

---

There are two ways to apply the chain rule through a graph:

**Forward mode** starts at the inputs and pushes derivatives forward. It computes `dx/dx = 1` and propagates through each operation. Good when you have few inputs and many outputs.

```
Forward mode: seed dx/dx = 1, propagate forward

  x = 2       (dx/dx = 1)
  a = x²      (da/dx = 2x = 4)
  y = sin(a)  (dy/dx = cos(a) · da/dx = cos(4) · 4 = -2.615)
```

**Reverse mode** starts at the output and pulls gradients backward. It computes `dy/dy = 1` and propagates through each operation in reverse. Good when you have many inputs and few outputs.

```
Reverse mode: seed dy/dy = 1, propagate backward

  y = sin(a)  (dy/dy = 1)
  a = x²      (dy/da = cos(a) = cos(4) = -0.654)
  x = 2       (dy/dx = dy/da · da/dx = -0.654 · 4 = -2.615)
```

Neural networks have millions of inputs (weights) and one output (loss). Reverse mode computes all gradients in one backward pass. This is why backpropagation uses reverse mode.


In [ ]:
import math# Reverse-mode example: y = sin(x^2), x = 2x = 2.0a = x ** 2y = math.sin(a)# Forward: valuesprint(f"x = {x}, a = x^2 = {a}, y = sin(a) = {y:.4f}")# Reverse: gradients (dy/dy = 1)dy_dy = 1.0dy_da = math.cos(a) * dy_dydy_dx = dy_da * (2 * x)print(f"dy/da = cos(a) = {dy_da:.4f}")print(f"dy/dx = dy/da · 2x = {dy_dx:.4f}")# Verify with numericalh = 1e-7y_plus = math.sin((x + h) ** 2)y_minus = math.sin((x - h) ** 2)num = (y_plus - y_minus) / (2 * h)print(f"Numerical dy/dx = {num:.4f}")print(f"Match: {abs(dy_dx - num) < 1e-5}")

In [ ]:
class Value:    def __init__(self, data, children=(), op=''):        self.data = data        self.grad = 0.0        self._backward = lambda: None        self._prev = set(children)        self._op = op    def __repr__(self):        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"    def __add__(self, other):        other = other if isinstance(other, Value) else Value(other)        out = Value(self.data + other.data, (self, other), '+')        def _backward():            self.grad += out.grad            other.grad += out.grad        out._backward = _backward        return out    def __mul__(self, other):        other = other if isinstance(other, Value) else Value(other)        out = Value(self.data * other.data, (self, other), '*')        def _backward():            self.grad += other.data * out.grad            other.grad += self.data * out.grad        out._backward = _backward        return out    def relu(self):        out = Value(max(0, self.data), (self,), 'relu')        def _backward():            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad        out._backward = _backward        return out    def backward(self):        topo = []; visited = set()        def build_topo(v):            if v not in visited:                visited.add(v)                for child in v._prev:                    build_topo(child)                topo.append(v)        build_topo(self)        self.grad = 1.0        for v in reversed(topo):            v._backward()# Manual verification: y = relu(x1 * x2 + 1), x1=2, x2=3x1 = Value(2.0)x2 = Value(3.0)a = x1 * x2          # a = 6.0b = a + Value(1.0)   # b = 7.0y = b.relu()          # y = 7.0y.backward()print(f"y = {y.data}")          # 7.0print(f"dy/dx1 = {x1.grad}")   # 3.0 (= x2)print(f"dy/dx2 = {x2.grad}")   # 2.0 (= x1)print("\nManual check: y = relu(x1*x2 + 1). Since x1*x2+1 = 7 > 0, relu is identity.")print("dy/dx1 = x2 = 3.  dy/dx2 = x1 = 2.  Engine matches.")

# More Operations for a Complete Engine

---

The basic Value class handles addition, multiplication, and relu. A real autograd engine needs more. Here are the operations needed to build neural networks:

| Operation | Backward rule | Used in |
|-----------|--------------|---------|
| `__sub__` | Reuses add + neg | Loss computation (pred − target) |
| `__pow__` | n · x^(n−1) | Polynomial activations, MSE (error²) |
| `__truediv__` | Reuses mul + pow(−1) | Normalization, learning rate scaling |
| `exp` | exp(x) · upstream | Softmax, log-likelihood |
| `log` | (1/x) · upstream | Cross-entropy loss, log probabilities |
| `tanh` | (1 − tanh²) · upstream | Classic activation function |

The clever part: `__sub__` and `__truediv__` are defined in terms of existing operations. They get correct gradients for free because the chain rule composes through the underlying add/mul/pow operations.


In [ ]:
import mathclass Value2:    def __init__(self, data, children=(), op=''):        self.data = data; self.grad = 0.0; self._backward = lambda: None        self._prev = set(children); self._op = op    def __add__(self, other):        other = other if isinstance(other, Value2) else Value2(other)        out = Value2(self.data + other.data, (self, other), '+')        def _backward(): self.grad += out.grad; other.grad += out.grad        out._backward = _backward; return out    def __mul__(self, other):        other = other if isinstance(other, Value2) else Value2(other)        out = Value2(self.data * other.data, (self, other), '*')        def _backward(): self.grad += other.data * out.grad; other.grad += self.data * out.grad        out._backward = _backward; return out    def __pow__(self, n):        out = Value2(self.data ** n, (self,), f'**{n}')        def _backward(): self.grad += n * (self.data ** (n - 1)) * out.grad        out._backward = _backward; return out    def exp(self):        e = math.exp(self.data); out = Value2(e, (self,), 'exp')        def _backward(): self.grad += e * out.grad        out._backward = _backward; return out    def tanh(self):        t = math.tanh(self.data); out = Value2(t, (self,), 'tanh')        def _backward(): self.grad += (1 - t ** 2) * out.grad        out._backward = _backward; return out    def backward(self):        topo = []; visited = set()        def bt(v):            if v not in visited:                visited.add(v)                for c in v._prev: bt(c)                topo.append(v)        bt(self)        self.grad = 1.0        for v in reversed(topo): v._backward()# Gradient check: d/dx(x^3) at x=2 should be 12x = Value2(2.0)y = x ** 3y.backward()print(f"Autodiff d/dx(x^3) at x=2: {x.grad:.6f}")print(f"Expected: 12.0")# tanh'(0) should be 1, tanh'(2) ≈ 0.0707for xv in [0.0, 2.0]:    xv_obj = Value2(xv)    yv = xv_obj.tanh()    yv.backward()    print(f"tanh'({xv}) = {xv_obj.grad:.6f}")

# Mini MLP from Scratch

---

With a complete Value class, you can build a neural network. No PyTorch. No NumPy. Just Values and the chain rule.

A `Neuron` computes `tanh(w1·x1 + w2·x2 + ... + b)`. A `Layer` is a list of neurons. An `MLP` stacks layers. Every weight is a `Value`, so calling `loss.backward()` propagates gradients to every parameter.


In [ ]:
import randomrandom.seed(42)class Neuron:    def __init__(self, n_inputs):        self.w = [Value2(random.uniform(-1, 1)) for _ in range(n_inputs)]        self.b = Value2(0.0)    def __call__(self, x):        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)        return act.tanh()    def parameters(self): return self.w + [self.b]class Layer:    def __init__(self, n_inputs, n_outputs):        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]    def __call__(self, x): return [n(x) for n in self.neurons]    def parameters(self): return [p for n in self.neurons for p in n.parameters()]class MLP:    def __init__(self, sizes):        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]    def __call__(self, x):        for layer in self.layers: x = layer(x)        return x[0] if len(x) == 1 else x    def parameters(self): return [p for layer in self.layers for p in layer.parameters()]# Train on XORmodel = MLP([2, 4, 1])  # 2 inputs, 4 hidden, 1 outputxs = [[0, 0], [0, 1], [1, 0], [1, 1]]ys = [-1, 1, 1, -1]  # XOR pattern (using -1/1 for tanh)for step in range(100):    preds = [model(x) for x in xs]    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))    for p in model.parameters(): p.grad = 0.0    loss.backward()    lr = 0.05    for p in model.parameters(): p.data -= lr * p.grad    if step % 20 == 0: print(f"step {step:3d}  loss = {loss.data:.4f}")print("\nPredictions after training:")for x, y in zip(xs, ys):    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")

# Gradient Checking

---

How do you know your autodiff is correct? Compare it against numerical derivatives. This is **gradient checking** — essential when implementing new operations.

```
def gradient_check(build_expr, x_val, h=1e-7):
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    autodiff_grad = x.grad
    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)
    diff = abs(autodiff_grad - numerical_grad)
    return autodiff_grad, numerical_grad, diff
```


In [ ]:
def gradient_check(build_expr, x_val, h=1e-7):    x = Value2(x_val)    y = build_expr(x)    y.backward()    ad = x.grad    y_plus = build_expr(Value2(x_val + h)).data    y_minus = build_expr(Value2(x_val - h)).data    num = (y_plus - y_minus) / (2 * h)    return ad, num, abs(ad - num)def expr(x):    return (x ** 3 + x * 2 + 1).tanh()ad, num, diff = gradient_check(expr, 0.5)print(f"Autodiff:  {ad:.8f}")print(f"Numerical: {num:.8f}")print(f"Difference: {diff:.2e}")print(f"Match (< 1e-5): {diff < 1e-5}")

# Verify Against PyTorch

---

Your engine computes the same result as PyTorch because the math is the same: reverse-mode autodiff via the chain rule.

```python
import torch
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
a = x1 * x2
b = a + 1.0
y = torch.relu(b)
y.backward()
print(f"PyTorch dy/dx1 = {x1.grad.item()}")  # 3.0
print(f"PyTorch dy/dx2 = {x2.grad.item()}")  # 2.0
```


# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What is reverse-mode autodiff?
> **Explanation:** Propagating gradients from output to input through a computational graph. One pass computes all gradients — ideal for neural nets with millions of weights and one loss.

**Q2:** For z = x·y, what is dz/dx?
> **Explanation:** y — the gradient w.r.t. x is scaled by the other input value.

**Q3:** Why gradient accumulation uses += not =?
> **Explanation:** When a value feeds into multiple operations, its total gradient is the sum of all incoming contributions.

**Q4:** What does gradient checking verify?
> **Explanation:** That your autodiff backward pass matches numerical finite-difference derivatives (catches bugs in new operations).



# Practice Exercises

---

Do these in order:

**Exercise 1:** Add `__pow__` to the Value class so you can compute `x ** n`. Verify that `d/dx(x^3)` at `x=2` equals `12.0`.

**Exercise 2:** Add `tanh` as an activation function. Verify that `tanh'(0) = 1` and `tanh'(2) ≈ 0.0707`.

**Exercise 3:** Build a computation graph for a single neuron: `y = relu(w1·x1 + w2·x2 + b)`. Compute all five gradients and verify against PyTorch.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Chain rule | "Multiply the derivatives" | The derivative of composed functions equals the product of each function's local derivative, evaluated at the right point. |
| Computational graph | "The network diagram" | A DAG where nodes are operations and edges carry values (forward) or gradients (backward). |
| Forward mode | "Push derivatives forward" | Autodiff propagating derivatives from inputs to outputs. One pass per input variable. |
| Reverse mode | "Backpropagation" | Autodiff propagating gradients from outputs to inputs. One pass per output variable. |
| Autograd | "Automatic gradients" | A system that records operations on values, builds a graph, and computes exact gradients via the chain rule. |
| Dual numbers | "Value plus derivative" | Numbers of the form a + b·ε (ε² = 0) that carry derivative info through arithmetic. |
| Topological sort | "Dependency order" | Ordering graph nodes so every node comes after all its dependencies. Required for correct gradient propagation. |
| Gradient accumulation | "Add, don't replace" | When a value feeds into multiple operations, its gradient is the sum of all incoming contributions. |
| Dynamic graph | "Define by run" | A computation graph rebuilt on every forward pass, allowing Python control flow inside models (PyTorch style). |
| Gradient checking | "Numerical verification" | Comparing autodiff gradients against numerical finite-difference gradients to verify correctness. |
| MLP | "Multi-layer perceptron" | A neural network with one or more hidden layers of neurons. |
| Neuron | "Weighted sum + activation" | Output = activation(w1·x1 + w2·x2 + ... + b). The weights and bias are learnable parameters. |


# Further Reading

---

- [3Blue1Brown: Backpropagation calculus](https://www.youtube.com/watch?v=tIeHLnjs5U8) — visual explanation of the chain rule in neural networks
- [PyTorch Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html) — how the real system works
- [Baydin et al., Automatic Differentiation in Machine Learning: a Survey](https://arxiv.org/abs/1502.05767) — comprehensive reference


# Next Steps

---

When you're ready, continue to Lesson 06: Probability and Distributions.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
